# CrisisLens: HumSet audit and baselines (v0.1)

Companion notebook for [MHKhan50/CrisisLens-AgentBench](https://github.com/MHKhan50/CrisisLens-AgentBench).
HumSet (Fekih et al., Findings of EMNLP 2022; Apache-2.0) is downloaded from the Hugging Face Hub at run time and not redistributed.

Steps: provenance manifest, dataset audit (languages, labels, duplicates, document leakage across splits), then a prior baseline and a TF-IDF + logistic-regression baseline on all five label sets, on the official split and on a document-grouped split, with document-level bootstrap intervals and per-language scores.

Kaggle settings: **CPU**, **Internet on**. *Save Version → Save & Run All*. About 1–1.5 hours. Outputs: `/kaggle/working/crisislens_v01_reports.zip`.

Humanitarian research only: not for targeting, surveillance, person-level profiling or automated aid decisions.

In [1]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("CrisisLens-AgentBench"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/CrisisLens-AgentBench.git"], check=True)
os.chdir("CrisisLens-AgentBench")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
DATA, REPORTS = "/kaggle/working/data/humset", "/kaggle/working/reports"
def cl(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "crisislens_agentbench.cli", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

e91fb5d HumSet loader: use the official 2.0.0 split files; reuse results only for the same data files

..............                                                           [100%]
=============================== warnings summary ===============================
tests/test_modeling.py::test_baseline_trains_and_evaluates
tests/test_pipeline.py::test_pipeline_preserves_provenance_and_review_flag
  /usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
    opt_res = optimize.minimize(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
14 passed, 2 warnings in 4.66s


CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q'], returncode=0)

## 1. Download with provenance (revision and SHA-256 of every file)

In [2]:
cl("humset-download", "--data-dir", DATA)


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 4 files: 100%|██████████| 4/4 [00:01<00:00,  2.28it/s]
{
  "dataset": "nlp-thedeep/humset",
  "version": "2.0.0",
  "revision": "0dcf8311cae5a45dee0ded3fea676a1551c1cd68",
  "licence": "Apache-2.0 (dataset card)",
  "citation": "Fekih et al., HumSet, Findings of EMNLP 2022",
  "files": {
    "train": {
      "path": "data/train_1_1.jsonl",
      "bytes": 138912205,
      "sha256": "fdc84512977ca2de48af33db27ed54da0f48ca3e06ae4db790aa9e0ded11ca1c"
    },
    "validation": {
      "path": "data/validation_1_1.jsonl",
      "bytes": 12182884,
      "sha256": "ddf86df1d2693dc2a5e1b5f9f1d821a08a9f3748b93bf6f14b052e1968324cfd"
    },
    "test": {
      "path": "data/test_1_1.jsonl",
      "bytes": 13133846,
      "sha256": "2484b6fa7f730e120203857dde2e63fc3b9c66f915ac9c7ee0990d6feefe619e"
 

## 2. Smoke test on a small subset

In [3]:
cl("humset-baselines", "--data-dir", DATA, "--reports", "/kaggle/working/smoke", "--limit", "800", "--n-boot", "50", "--tasks", "sectors")
import shutil; shutil.rmtree("/kaggle/working/smoke", ignore_errors=True)

train: data/train_1_1.jsonl rows=129268 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', 'displaced', 'affected', 'severity', 'age', 'gender', 'non_displaced', 'specific_needs_groups']
validation: data/validation_1_1.jsonl rows=11940 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', 'displaced', 'affected', 'severity', 'age', 'gender', 'non_displaced', 'specific_needs_groups']
test: data/test_1_1.jsonl rows=12896 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', '

## 3. Dataset audit

In [4]:
cl("humset-audit", "--data-dir", DATA, "--reports", REPORTS)

train: data/train_1_1.jsonl rows=129268 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', 'displaced', 'affected', 'severity', 'age', 'gender', 'non_displaced', 'specific_needs_groups']
validation: data/validation_1_1.jsonl rows=11940 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', 'displaced', 'affected', 'severity', 'age', 'gender', 'non_displaced', 'specific_needs_groups']
test: data/test_1_1.jsonl rows=12896 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', '

## 4. Baselines: official split and document-grouped split

In [5]:
cl("humset-baselines", "--data-dir", DATA, "--reports", REPORTS)
import shutil, json
shutil.copy(DATA + "/manifest.json", REPORTS + "/manifest.json")
shutil.make_archive("/kaggle/working/crisislens_v01_reports", "zip", REPORTS)
print(open(REPORTS + "/audit.md").read()); print(open(REPORTS + "/baselines.md").read())

train: data/train_1_1.jsonl rows=129268 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', 'displaced', 'affected', 'severity', 'age', 'gender', 'non_displaced', 'specific_needs_groups']
validation: data/validation_1_1.jsonl rows=11940 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', 'displaced', 'affected', 'severity', 'age', 'gender', 'non_displaced', 'specific_needs_groups']
test: data/test_1_1.jsonl rows=12896 columns=['entry_id', 'excerpt', 'lead_id', 'geo_location', 'lang', 'n_tokens', 'project_title', 'created_at', 'document', 'source_title', 'author_title', 'sectors', 'subpillars_2d', 'subpillars_1d', 'pillars_1d', 'pillars_2d', '